In [6]:
import xarray as xr
import pystac_client
import fsspec
import planetary_computer
import pandas as pd

In [2]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [3]:
asset = catalog.get_collection("gpm-imerg-hhr").assets["zarr-abfs"]

In [12]:
fs = fsspec.get_mapper(asset.href, **asset.extra_fields["xarray:storage_options"])
ds = xr.open_zarr(fs, **asset.extra_fields["xarray:open_kwargs"])

In [7]:
meta = pd.read_csv('./Data/metadata/IMN_stations_rew.csv')

In [10]:
meta = meta[meta['percentage'] <= 10]

In [18]:
ds = ds['precipitationCal']

In [22]:
# Extract specific latitude and longitude values from the DataFrame
Lat = meta['lat'].values  # Replace 'lat' with your latitude column name
Lon = meta['lon'].values  # Replace 'lon' with your longitude column name

In [23]:
# Select the nearest latitude and longitude points from xarray dataset
selected_data = ds.sel(
    lat=Lat,
    lon=Lon,
    method='nearest'
)

/Users/maureenfonseca/opt/anaconda3/lib/python3.9/site-packages/xarray/core/indexing.py:1449: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  value = value[(slice(None),) * axis + (subkey,)]


In [27]:
selected_data = selected_data.resample(time='1H').mean()

In [ ]:
t = selected_data.sel(lat=11.05, lon=-84.75).to_dataframe()